# Phase 0: Setup & Foundations

**Enterprise Agentic RAG System**

This notebook validates the Phase 0 foundations:
- Configuration loading (YAML + Pydantic + .env)
- Structured logging
- PDF ingestion (page-aware)
- Configurable chunking
- Basic indexing + retrieval scaffolding

Run this after following the setup instructions in README.md.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

# Make src importable when running from notebooks/
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)

## 1. Load Configuration & Logging

In [ ]:
from src.config import get_settings, settings
from src.logging_config import logger, setup_logging

print("RAG_ENV:", settings.rag_env)
print("Primary PDF:", settings.document.primary_pdf)
print("Chunk size (tokens):", settings.chunking.chunk_size_tokens)
print("Embedding model:", settings.indexing.embedding.model)
print("LLM model:", settings.generation.llm.model)

In [ ]:
# Re-setup logging if needed (usually already configured on import)
setup_logging()
logger.info("Phase 0 notebook started")

## 2. Ingestion — Load the AI Agents Guidebook

In [ ]:
from src.ingestion.pipeline import IngestionPipeline

pipeline = IngestionPipeline()
doc = pipeline.run()   # Uses config to locate data/raw/ai_agents_guidebook.pdf

print(f"Document: {doc.doc_id}")
print(f"Total pages: {doc.total_pages}")
print(f"Total characters: {sum(p['char_count'] for p in doc.pages):,}")

# Show a sample from page 5
if doc.pages:
    sample_page = doc.pages[min(4, len(doc.pages)-1)]
    print("\n--- Sample from page", sample_page['page_num'], "---")
    print(sample_page['text'][:800])

## 3. Chunking

In [ ]:
from src.chunking.splitter import TextChunker

chunker = TextChunker()

# Build a simple page_map for page-range assignment
page_map = [(p['page_num'], p['text']) for p in doc.pages]

full_text = doc.get_full_text()
chunks = chunker.chunk(full_text, doc_id=doc.doc_id, page_map=page_map)

print(f"Total chunks: {len(chunks)}")
print("First chunk preview:")
print(chunks[0].text[:600])
print("\nChunk metadata (first 3):")
for c in chunks[:3]:
    print(f"  {c.chunk_id} | pages {c.page_start}-{c.page_end} | tokens ~{c.token_count}")

## 4. (Optional) Quick Indexing + Retrieval Smoke Test

**Warning:** This will call the OpenAI Embedding API and create a local Chroma collection.
Make sure your `OPENAI_API_KEY` is set in `.env`.

In [ ]:
# Uncomment the block below to run a real embedding + retrieval test

# from src.indexing.indexer import Indexer
# from src.retrieval.retriever import Retriever

# indexer = Indexer()
# indexer.index_chunks(chunks[:50])   # small slice for quick test

# retriever = Retriever()
# results = retriever.retrieve("What are the key principles of building reliable AI agents?", top_k=5)

# for r in results:
#     print(f"[{r.page_start}] {r.text[:200]}...\n")

## 5. Generation (Baseline) — commented for safety

Uncomment once you have indexed at least some chunks.

In [ ]:
# from src.generation.generator import RAGGenerator

# generator = RAGGenerator()
# answer = generator.generate(
#     query="Summarize the main components of an agentic RAG system.",
#     retrieved=results   # from previous cell
# )
# print(answer)

## Next Steps (Phase 0 → Phase 1)

1. Run full ingestion + chunking on the entire guidebook.
2. Index the full set of chunks.
3. Build a simple evaluation harness.
4. Move to `src/retrieval` improvements and generation quality work.

Happy building!